# **Individual Analysis of Tripartite Networks**

# Import Libraries and Configurations

In [1]:
import os
import sys

import networkx as nx
import pandas as pd

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    DATA_SUBDIRS,
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
)

# Remove the HER2-enriched group from the analysis
GROUPS = DATA_SUBDIRS.copy()
GROUPS.remove('her2-enriched')

# Metrics calculated in the individual analysis
METRICS = [
    # 'degree', 
    # 'betweenness_centrality', 
    # 'closeness_centrality', 
    # 'degree_centrality', 
    # 'redundancy_coefficient', 
    'pathway_reach',
    'functional_impact',
]

# Functions

## Network Object

In [2]:
def create_tripartite_network(dir_path):
    # Create DataFrames to represent the microRNA-mRNA network
    int_edges_file = NETWORK_FILES['interaction-edges']
    df_int_edges = pd.read_csv(
        os.path.join(dir_path, int_edges_file)
    )
    int_nodes_file = (
        f'{(NETWORK_FILES['interaction-nodes']).replace('.csv', '')}' 
        + '-with-metrics.csv'
    )
    df_int_nodes = pd.read_csv(
        os.path.join(dir_path, int_nodes_file)
    )

    # Create DataFrames to represent the mRNA-pathway network
    mem_edges_file = 'membership-network-edges.csv'
    df_mem_edges = pd.read_csv(
        os.path.join(dir_path, mem_edges_file)
    )
    mem_nodes_file = 'membership-network-nodes-with-metrics.csv'
    df_mem_nodes = pd.read_csv(
        os.path.join(dir_path, mem_nodes_file)
    )

    # Create DataFrames to represent the microRNA-mRNA-pathway network
    df_nodes = \
        pd.concat(objs=[df_int_nodes, df_mem_nodes]) \
        .drop_duplicates(ignore_index=True)
    df_edges = \
        pd.concat(objs=[df_int_edges, df_mem_edges])

    # Create the microRNA-mRNA-pathway network object
    T = nx.from_pandas_edgelist(
        df=df_edges,
        source='source',
        target='target',
        edge_attr=True,
        create_using=nx.DiGraph(),
    )

    # Assign partition attribute
    attr_dict = df_nodes.set_index('id')['type'].to_dict()
    nx.set_node_attributes(
        G=T, values=attr_dict, name='partition'
    )

    return T, df_edges, df_nodes

## Topological Analysis

### Paths Between MicroRNAs and Pathways

In [3]:
def identify_mir_to_pathway_paths(T, mir_set):
    # Create a dictionary to store the paths
    paths = dict()

    # Iterate over the microRNAs
    for mir in mir_set:
        # Initialize the microRNA path list
        paths[mir] = list()

        # Iterate over the neighbors of microRNAs
        for gene in T.neighbors(mir):
            # Iterate over the neighbors of the neighbors of microRNAs
            for pathway in T.neighbors(gene):
                # Store the other two nodes in the path
                paths[mir].append((gene, pathway))

        # Add the paths as an attribute of the microRNA nodes
        nx.set_node_attributes(
            G=T, values=paths, name='paths'
        )

    return paths

### Pathway Reach Calculation

In [4]:
def compute_pathway_reach(T, paths, pathway_set):
    # Create a list to store the metric values
    pathway_reach = dict()

    # Compute the number of pathway nodes in the network
    total_pathways = len(pathway_set)

    # Iterate over the microRNAs
    for mir in paths.keys():
        # Check if the microRNA reaches any pathway
        if len(paths[mir]) == 0:
            pathway_reach[mir] = 0
        else:
            # Create a list to store the reached pathways
            reached_pathways = list()

            # Iterate over the paths
            for _, pathway in paths[mir]:
                reached_pathways.append(pathway)

            # Remove the repeated patways
            reached_pathways = set(reached_pathways)

            # Store the pathway reach value in the dictionary
            value = len(reached_pathways) / total_pathways
            pathway_reach[mir] = value

    # Add the pathway reach as an attribute of the nodes
    nx.set_node_attributes(
        G=T, values=pathway_reach, name='pathway_reach'
    )

### Functional Impact Calculation

In [5]:
def compute_functional_impact(T, paths):
    # Create a list to store the metric values
    functional_impact = dict()

    # Iterate over the microRNAs
    for mir in paths.keys():
        # Initialize the metric value
        functional_impact[mir] = 0.0

        # Check if the microRNA reaches any pathway
        if len(paths[mir]) > 0:
            # Iterate over the paths
            for gene, pathway in paths[mir]:
                # Interaction correlation
                w1 = abs(T[mir][gene]['correlation'])

                # Interaction confidence
                w2 = 1 - (T[mir][gene]['qvalue'])

                # Membership relevance
                # w3 = - np.log10(T[gene][pathway]['qvalue'])
                w3 = 1 - (T[gene][pathway]['qvalue'])
                
                # Store the pathway reach value in the dictionary
                impact = w1 * w2 * w3
                functional_impact[mir] += impact
            
            # Normalize the metric value for comparisons
            functional_impact[mir] /= len(paths[mir])
    
    # Add the functional impact as an attribute of the nodes
    nx.set_node_attributes(
        G=T, values=functional_impact, name='functional_impact'
    )

### Compute Metrics

In [6]:
def compute_metrics(T):
    # Define the nodes in the microRNA and pathway partitions
    mir_set = [
        n for n, d in T.nodes(data=True) 
        if d['partition'] == 'MicroRNA'
    ]
    pathway_set = [
        n for n, d in T.nodes(data=True) 
        if d['partition'] == 'Pathway'
    ]

    # Detect the paths between microRNAs and pathways
    paths = identify_mir_to_pathway_paths(T, mir_set)

    # Compute the pathway reach of microRNAs
    compute_pathway_reach(T, paths, pathway_set)

    # Compute the functional impact of microRNAs
    compute_functional_impact(T, paths)

    # Store the node attributes in a DataFrame
    nodes = dict(T.nodes(data=True))
    df_metrics = pd.DataFrame.from_dict(data=nodes, orient='index') \
        .reset_index() \
        .drop(columns=['partition']) \
        .rename(columns={'index': 'id'})

    return df_metrics

## Analysis Report

### Print the Description of the Metric Values

In [7]:
def print_metric_values_description(df_metrics, metric, nodes_to_rank=5):
    # Print the name of the metric as a title
    metric_name = metric.replace('_', ' ')
    print(f'\n##### {metric_name.title()} #####\n')

    # Sort the nodes according to the parameters
    df_sorted = df_metrics \
        .sort_values(by=metric, ascending=False) \
        .dropna(subset=metric) \
        .reset_index(drop=True) \
        [['label', metric]]
    
    # Print the descriptive statistics
    print(f'> descriptive statistics: \n{df_sorted.describe()}')
    
    # Print the top and bottom ranked nodes
    for rank in ['top', 'bottom']:
        if rank == 'top':
            df_nodes = df_sorted.head(nodes_to_rank)
        else:
            df_nodes = df_sorted.tail(nodes_to_rank)

        print(
            f'\n> {rank}-{nodes_to_rank} nodes in '
            + f'descending order: \n{df_nodes}'
        )

### MicroRNA Report

In [8]:
def mir_partition_report(df_nodes):
    # Select the partition nodes
    partition = 'MicroRNA'
    df_partition = df_nodes.query('type == @partition')
    print(f'===== {partition}s =====')

    # Print the description of the metric values
    for metric in METRICS:
        print_metric_values_description(
            df_metrics=df_partition,
            metric=metric,
        )

## Individual Network Analysis

In [9]:
def individual_analysis(group):
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create the network object
    T, df_edges, df_nodes = create_tripartite_network(dir_path)

    # Store the DataFrames of nodes with metric values
    edges_file = 'tripartite-network-edges.csv'
    df_edges.to_csv(os.path.join(dir_path, edges_file), index=False)
    nodes_file = 'tripartite-network-nodes.csv'
    df_nodes.to_csv(os.path.join(dir_path, nodes_file), index=False)

    # Perform the topological analysis of the network
    df_metrics = compute_metrics(T)

    # Add the metric values in the node's DataFrame
    df_nodes = pd.merge(
        left=df_nodes, right=df_metrics, how='left', on='id'
    )

    # Store the DataFrame of nodes with metric values
    file_name = f'{nodes_file.replace('.csv', '')}-with-metrics.csv'
    df_nodes.to_csv(os.path.join(dir_path, file_name), index=False)

    return df_edges, df_nodes

# Analyze the Networks

In [10]:
# Create a dictionary to store DataFrames related to network analysis
analyzed_networks = dict()
analyzed_networks['edges'] = pd.DataFrame()
analyzed_networks['nodes'] = pd.DataFrame()

# Iterate over the groups
for group in GROUPS:    
    # Perform individual network analysis
    df_edges, df_nodes = individual_analysis(group)

    # Add group name column
    df_edges['group'] = group
    df_nodes['group'] = group
    
    # Concatenate the edges and nodes of the group to the others
    analyzed_networks['edges'] = pd.concat(
        objs=[analyzed_networks['edges'], df_edges], 
        ignore_index=True
    )
    analyzed_networks['nodes'] = pd.concat(
        objs=[analyzed_networks['nodes'], df_nodes], 
        ignore_index=True
    )

# Basal-like

In [11]:
# Define the group name and the DataFrames associated with it
group = 'basal-like'
df_edges = analyzed_networks['edges'].query('group == @group')
df_nodes = analyzed_networks['nodes'].query('group == @group')

In [12]:
# Print the analysis report for the microRNA nodes
mir_partition_report(df_nodes)

===== MicroRNAs =====

##### Pathway Reach #####

> descriptive statistics: 
       pathway_reach
count      58.000000
mean        0.161804
std         0.232646
min         0.000000
25%         0.000000
50%         0.076923
75%         0.288462
max         1.000000

> top-5 nodes in descending order: 
             label  pathway_reach
0    hsa-miR-96-5p       1.000000
1  hsa-miR-106b-5p       0.846154
2    hsa-let-7e-5p       0.846154
3   hsa-miR-30b-5p       0.538462
4  hsa-miR-106a-5p       0.538462

> bottom-5 nodes in descending order: 
              label  pathway_reach
53  hsa-miR-151a-3p            0.0
54    hsa-miR-21-5p            0.0
55   hsa-miR-424-5p            0.0
56  hsa-miR-193b-3p            0.0
57     hsa-miR-378c            0.0

##### Functional Impact #####

> descriptive statistics: 
       functional_impact
count          58.000000
mean            0.207202
std             0.190001
min             0.000000
25%             0.000000
50%             0.344274
75%      

# Luminal A

In [13]:
# Define the group name and the DataFrames associated with it
group = 'luminal-a'
df_edges = analyzed_networks['edges'].query('group == @group')
df_nodes = analyzed_networks['nodes'].query('group == @group')

In [14]:
# Print the analysis report for the microRNA nodes
mir_partition_report(df_nodes)

===== MicroRNAs =====

##### Pathway Reach #####

> descriptive statistics: 
       pathway_reach
count      44.000000
mean        0.259091
std         0.341214
min         0.000000
25%         0.000000
50%         0.000000
75%         0.450000
max         1.000000

> top-5 nodes in descending order: 
            label  pathway_reach
0   hsa-miR-96-5p            1.0
1     hsa-miR-429            1.0
2  hsa-miR-33a-5p            1.0
3  hsa-miR-33b-5p            0.8
4   hsa-miR-17-5p            0.8

> bottom-5 nodes in descending order: 
              label  pathway_reach
39  hsa-miR-196a-5p            0.0
40   hsa-miR-27a-3p            0.0
41    hsa-let-7c-5p            0.0
42  hsa-miR-365b-3p            0.0
43  hsa-miR-365a-3p            0.0

##### Functional Impact #####

> descriptive statistics: 
       functional_impact
count          44.000000
mean            0.150787
std             0.168216
min             0.000000
25%             0.000000
50%             0.000000
75%            

# Luminal B

In [15]:
# Define the group name and the DataFrames associated with it
group = 'luminal-b'
df_edges = analyzed_networks['edges'].query('group == @group')
df_nodes = analyzed_networks['nodes'].query('group == @group')

In [16]:
# Print the analysis report for the microRNA nodes
mir_partition_report(df_nodes)

===== MicroRNAs =====

##### Pathway Reach #####

> descriptive statistics: 
       pathway_reach
count     105.000000
mean        0.127273
std         0.193642
min         0.000000
25%         0.000000
50%         0.030303
75%         0.181818
max         0.818182

> top-5 nodes in descending order: 
             label  pathway_reach
0   hsa-miR-185-5p       0.818182
1  hsa-miR-200c-3p       0.787879
2   hsa-miR-26a-5p       0.727273
3   hsa-miR-15b-5p       0.696970
4   hsa-miR-18a-5p       0.636364

> bottom-5 nodes in descending order: 
               label  pathway_reach
100  hsa-miR-200a-3p            0.0
101  hsa-miR-374b-5p            0.0
102    hsa-let-7f-5p            0.0
103   hsa-miR-30e-5p            0.0
104  hsa-miR-146b-5p            0.0

##### Functional Impact #####

> descriptive statistics: 
       functional_impact
count         105.000000
mean            0.187834
std             0.161889
min             0.000000
25%             0.000000
50%             0.293433
75%

# Normal

In [17]:
# Define the group name and the DataFrames associated with it
group = 'normal'
df_edges = analyzed_networks['edges'].query('group == @group')
df_nodes = analyzed_networks['nodes'].query('group == @group')

In [18]:
# Print the analysis report for the microRNA nodes
mir_partition_report(df_nodes)

===== MicroRNAs =====

##### Pathway Reach #####

> descriptive statistics: 
       pathway_reach
count     166.000000
mean        0.216250
std         0.268412
min         0.000000
25%         0.010684
50%         0.081197
75%         0.356838
max         0.982906

> top-5 nodes in descending order: 
             label  pathway_reach
0   hsa-miR-20b-5p       0.982906
1    hsa-miR-93-5p       0.957265
2  hsa-miR-106a-5p       0.931624
3   hsa-miR-15a-5p       0.923077
4   hsa-miR-139-5p       0.880342

> bottom-5 nodes in descending order: 
              label  pathway_reach
161  hsa-miR-34a-5p            0.0
162  hsa-miR-33a-5p            0.0
163  hsa-miR-758-3p            0.0
164  hsa-miR-92b-3p            0.0
165   hsa-miR-32-5p            0.0

##### Functional Impact #####

> descriptive statistics: 
       functional_impact
count         166.000000
mean            0.340333
std             0.172586
min             0.000000
25%             0.335243
50%             0.372084
75%      